# 🧼 Phase 2.5 – Data Cleaning & Preparation for EDA

**Notebook:** `02_data_cleaning_and_preparation.ipynb`  
**Objective:** Clean and format the extended GeoDataFrame (`gdf_ca_extended_v3`) before conducting exploratory data analysis. Ensure consistency in variable types, naming, and missing values.

---

### 📌 Cleaning Goals

- Inspect and group columns by data type
- Drop unnecessary metadata and redundant fields
- Rename columns for consistency and readability
- Standardize percentage and distance values
- Handle missing data appropriately (e.g., FEMA aid, fault orientation)
- Prepare final cleaned GeoDataFrame for EDA

This notebook produces a **cleaned dataset** ready for analysis and modeling.

✅ Output file: `gdf_ca_cleaned.geojson`  
📁 Saved to: `../output/gdf_ca_cleaned.geojson`

---

### 🧾 Source Data

Cleaned from the following original inputs:

- `gdf_ca_extended_v3.geojson` — Merged dataset from Phase 2  
- `earthquake_zones_by_county.xlsx` — EQ Zone & Region corrections  
- Fault intersection and distance metrics — from Phase 3 fault-line integration

---

## 📦 Load Packages


In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np
import ast
from collections import defaultdict


---
## 📂 Data Loading – Load Final Merged Dataset


In [5]:
# Load the extended dataset from previous phase
gdf = gpd.read_file("../output/gdf_ca_extended_v3.geojson")

# Load and merge faults list from temporary csv
faults_list_col = pd.read_csv("../output/_temp_faults_list.csv")
faults_list_col["faults_list"] = faults_list_col["faults_list"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)
gdf = gdf.merge(faults_list_col, on="county", how="left")

gdf.head(3)


Skipping field faults_list: unsupported OGR type: 5


,county,GEOID,STATEFP,COUNTYFP,NAMELSAD,LSAD,total_population,median_income,age_0_17,age_18_64,...,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,faults_list
0,Sierra,06091,06,091,Sierra County,06,2731,60000,349.0,1440.0,...,None,None,None,A,2468694578,23299110,+39.5769252,-120.5219926,"POLYGON ((-120.55587 39.50874, -120.55614 39.5...","[Mohawk Valley fault zone, Polaris fault, Dog ..."
1,Sacramento,06067,06,067,Sacramento County,06,1584047,88724,365667.0,981913.0,...,472,40900,None,A,2500063005,75323439,+38.4501363,-121.3443291,"POLYGON ((-121.43991 38.25553, -121.44002 38.2...",[]
2,Santa Barbara,06083,06,083,Santa Barbara County,06,443975,95977,99608.0,272385.0,...,None,42200,None,A,7080874935,2729198796,+34.5366774,-120.0383645,"MULTIPOLYGON (((-120.58226 34.10752, -120.5790...","[Santa Cruz Island fault zone, Red Mountain fa..."


---
## 🧹 Data Cleaning & Formatting

Clean and format the extended GeoDataFrame to prepare it for exploratory analysis and modeling.  
Steps include:  
- Inspect & Group Columns by Type
- Drop unnecessary metadata columns  
- Standardize column names  
- Handle missing values  
- Round percentage and distance values  
- Fix data types  
- Export cleaned dataset  

---
### Inspect and Group Columns by Data Type


In [7]:
# Basic overview
print("Shape:", gdf.shape)
print("CRS:", gdf.crs)


Shape: (58, 74)
CRS: EPSG:4269


In [8]:
# Group columns by data type
dtype_groups = defaultdict(list)
for col, dtype in gdf.dtypes.items():
    dtype_groups[str(dtype)].append(col)

for dtype, cols in dtype_groups.items():
    print(f"\n📦 {dtype} ({len(cols)} columns):")
    print(cols)



📦 object (20 columns):
['county', 'GEOID', 'STATEFP', 'COUNTYFP', 'NAMELSAD', 'LSAD', 'EQ_Zone', 'Region', 'fault_orientation', 'COUNTYNS', 'GEOIDFQ', 'CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'INTPTLAT', 'INTPTLON', 'faults_list']

📦 int32 (15 columns):
['total_population', 'median_income', 'below_poverty', 'total_housing_units', 'built_pre_1980', 'built_1980_1999', 'built_2000_up', 'total_households', 'owner_occupied', 'renter_occupied', 'median_home_value', 'eq_policies', 'Total_Direct_Liability_2018', 'Direct_Insured_PML_2018', 'n_faults_intersecting']

📦 float64 (35 columns):
['age_0_17', 'age_18_64', 'age_65_up', 'age_0_17_pct', 'age_18_64_pct', 'age_65_up_pct', 'below_poverty_pct', 'white_alone_pct', 'black_alone_pct', 'american_indian_alone_pct', 'asian_alone_pct', 'native_hawaiian_alone_pct', 'hispanic_pct', 'owner_pct', 'renter_pct', 'for_rent_pct', 'rented_not_occupied_pct', 'for_sale_pct', 'sold_not_occupied_pct', 'seasonal_pct', 'migrant_pct', 'other_

---
### Drop Metadata Columns


In [10]:
# Drop unnecessary metadata columns
meta_cols = [
    'GEOID', 'STATEFP', 'COUNTYFP', 'NAMELSAD', 'LSAD', 
    'COUNTYNS', 'GEOIDFQ', 'CLASSFP', 'MTFCC', 'CSAFP', 
    'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'INTPTLAT', 'INTPTLON'
]
gdf_clean = gdf.drop(columns=meta_cols)
print(f"\nDropped {len(meta_cols)} metadata columns → Remaining: {gdf_clean.shape[1]}")



Dropped 15 metadata columns → Remaining: 59


---
### Standardize Column Names


In [12]:
# Rename columns for clarity 
gdf_clean.rename(columns={
    "ALAND": "land_area_m2",
    "AWATER": "water_area_m2",
    "EQ_Zone": "eq_zone",
    "Region": "region",
    "Total_Direct_Liability_2018": "total_direct_liability_2018",
    "Direct_Insured_PML_2018": "direct_insured_pml_2018",
    "Modeled_Loss_Ratio_2018_pct": "modeled_loss_ratio_2018_pct",
    "New_Voluntary": "new_voluntary",
    "Renewed_Voluntary": "renewed_voluntary",
    "New_FAIR": "new_fair",
    "Renewed_FAIR": "renewed_fair",
    "Total_Homeowners_Policies": "total_homeowners_policies"
}, inplace=True)


# Reorder columns — move 'faults_list' after 'n_faults_intersecting'
cols = list(gdf_clean.columns)
cols.remove("faults_list")
insert_idx = cols.index("n_faults_intersecting") + 1
cols.insert(insert_idx, "faults_list")
gdf_clean = gdf_clean[cols]

gdf_clean.columns


Index(['county', 'total_population', 'median_income', 'age_0_17', 'age_18_64',
       'age_65_up', 'age_0_17_pct', 'age_18_64_pct', 'age_65_up_pct',
       'below_poverty', 'below_poverty_pct', 'white_alone_pct',
       'black_alone_pct', 'american_indian_alone_pct', 'asian_alone_pct',
       'native_hawaiian_alone_pct', 'hispanic_pct', 'total_housing_units',
       'built_pre_1980', 'built_1980_1999', 'built_2000_up',
       'total_households', 'owner_occupied', 'renter_occupied', 'owner_pct',
       'renter_pct', 'for_rent_pct', 'rented_not_occupied_pct', 'for_sale_pct',
       'sold_not_occupied_pct', 'seasonal_pct', 'migrant_pct',
       'other_vacant_pct', 'median_home_value', 'eq_policies', 'eq_takeup_pct',
       'eq_zone', 'region', 'total_direct_liability_2018',
       'direct_insured_pml_2018', 'modeled_loss_ratio_2018_pct',
       'new_voluntary', 'renewed_voluntary', 'new_fair', 'renewed_fair',
       'total_homeowners_policies', 'homeowners_coverage_pct',
       'fema_fund

---
### Handle Missing Values


In [14]:
# Count missing values
missing_counts = gdf_clean.isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
display(missing_counts.to_frame(name="Missing Values"))


,Missing Values
fema_funding_earthquake,50
mean_fault_azimuth_deg,6


In [15]:
# Handle missing values
gdf_clean["fema_funding_earthquake"] = gdf_clean["fema_funding_earthquake"].fillna(0)
# Leave mean_fault_azimuth_deg as NaN – it reflects counties with no faults


---
### Round Percentages & Distance Columns


In [17]:
# Round percentage columns to 1 decimal
pct_cols = [col for col in gdf_clean.columns if col.endswith("_pct")]
gdf_clean[pct_cols] = gdf_clean[pct_cols].round(1)

# Round distances & angles to 2 decimals
dist_cols = [
    "centroid_to_fault_km", "nearest_fault_km",
    "fault_length_km", "mean_fault_azimuth_deg"
]
gdf_clean[dist_cols] = gdf_clean[dist_cols].round(2)


---
### Fix Data Types


In [19]:
cols_to_int = [
    'age_0_17', 'age_18_64', 'age_65_up',
    'new_voluntary', 'renewed_voluntary', 'new_fair', 'renewed_fair',
    'total_homeowners_policies'
]
gdf_clean[cols_to_int] = gdf_clean[cols_to_int].round().astype(int)

# Convert all int32 to int64
int32_cols = gdf_clean.select_dtypes("int32").columns
gdf_clean[int32_cols] = gdf_clean[int32_cols].astype("int64")

# Quick type verification
print("\nData types after cleaning:")
print(gdf_clean.dtypes.value_counts())



Data types after cleaning:
float64     27
int64       25
object       5
bool         1
geometry     1
Name: count, dtype: int64


---
### Export cleaned dataset

- `gdf_ca_cleaned.parquet`: full GeoDataFrame with lists preserved (for analysis)
- `gdf_ca_cleaned.geojson`: flattened version with string-based fault list (for GIS tools)


In [21]:
# Ensure 'faults_list' is in list format (not np.ndarray)
gdf_clean["faults_list"] = gdf_clean["faults_list"].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)

# Save 1: Parquet (preserves full types, best for EDA/modeling)
gdf_clean.to_parquet("../output/gdf_ca_cleaned.parquet", index=False)
print("✅ Saved gdf_ca_cleaned.parquet (full fidelity)")

# Save 2: GeoJSON (flatten 'faults_list' to string to avoid OGR issues)
gdf_geojson = gdf_clean.copy()
gdf_geojson["faults_list_str"] = gdf_geojson["faults_list"].apply(
    lambda L: ", ".join(map(str, L)) if isinstance(L, list) else ""
)
gdf_geojson["faults_list"] = gdf_geojson["faults_list_str"]
gdf_geojson.drop(columns=["faults_list_str"], inplace=True)
gdf_geojson.to_file("../output/gdf_ca_cleaned.geojson", driver="GeoJSON")
print("✅ Saved gdf_ca_cleaned.geojson (flattened for export)")


✅ Saved gdf_ca_cleaned.parquet (full fidelity)
✅ Saved gdf_ca_cleaned.geojson (flattened for export)


---

### 🖋️ Author & Attribution

This notebook is part of the **Earthquake Insurance Risk Analysis in California** project.  
Created and maintained by **Anna Piterskaya**  
Aspiring Data Scientist | Geospatial & Risk Modeling | [GitHub: annapiter](https://github.com/annapiter)

© 2025 Anna Piterskaya. All rights reserved.

📄 This work is licensed under the **Creative Commons Attribution-NonCommercial 4.0 International License**.  
You may share or adapt this work **with attribution** and **non-commercial use only**.  
🔗 [View License Details](https://creativecommons.org/licenses/by-nc/4.0/)
